# 04 — Continuous Evaluation and Observability

**Track:** Intermediate · **Stage:** Evaluation

Offline evaluation (using Golden Datasets and Ragas) proves your system works *before* deployment. **Continuous Evaluation** (Observability) proves your system continues to work *after* deployment.

When a user asks a question in production, you don't have a "Ground Truth Answer" to compare against. Instead, you must capture the entire trace of the interaction and run asynchronous evaluators over live traffic.

## Setup: Simulating LangSmith

Platforms like **LangSmith**, **Phoenix**, and **Datadog** automatically capture execution traces via callbacks. We will simulate this using LangChain's callback system.

In [ ]:
# !pip install langchain langchain-core

from typing import Any, Dict, List
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms.fake import FakeListLLM
from langchain_core.runnables import RunnablePassthrough

## 1. Capturing the Production Trace

We need to capture the exact question, the exact retrieved documents, and the final generated answer for every single request.

In [ ]:
class TraceCaptureCallback(BaseCallbackHandler):
    def __init__(self):
        self.traces = []
        self.current_trace = {}

    def on_chain_start(self, serialized: Dict[str, Any], inputs: Dict[str, Any], **kwargs: Any) -> Any:
        if "question" in inputs:
            self.current_trace["question"] = inputs["question"]

    def on_retriever_end(self, documents, **kwargs: Any) -> Any:
        self.current_trace["context"] = "\n".join([doc.page_content for doc in documents])

    def on_llm_end(self, response, **kwargs: Any) -> Any:
        self.current_trace["answer"] = response.generations[0][0].text
        
    def on_chain_end(self, outputs: Dict[str, Any], **kwargs: Any) -> Any:
        if "answer" in self.current_trace:
            self.traces.append(self.current_trace.copy())
            self.current_trace = {}

tracer = TraceCaptureCallback()

## 2. The Live Pipeline

Let's run a mock pipeline and capture the traces.

In [ ]:
from langchain_core.documents import Document

def mock_retrieve(query: str):
    if "water" in query.lower():
        return [Document(page_content="Water damage is capped at $50k.")]
    return [Document(page_content="General policy info.")]

prompt = ChatPromptTemplate.from_template("Context: {context}\nQuestion: {question}")
llm = FakeListLLM(responses=["The cap is $50k.", "I don't know."])
chain = {"context": mock_retrieve, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()

# Simulating live traffic
chain.invoke("What is the water damage cap?", config={"callbacks": [tracer]})
chain.invoke("What is the wind damage cap?", config={"callbacks": [tracer]})

print("--- Captured Production Traces ---")
for i, trace in enumerate(tracer.traces):
    print(f"\nTrace {i+1}:")
    print(f"  Question: {trace.get('question')}")
    print(f"  Context:  {trace.get('context')}")
    print(f"  Answer:   {trace.get('answer')}")

## 3. Asynchronous Online Evaluation

In LangSmith, you can configure an **Evaluator** that runs asynchronously over a random sample (e.g., 5%) of your production traces. 

Since we don't have Ground Truth, we can only evaluate reference-free metrics, such as:
1. **Faithfulness** (Does the answer hallucinate beyond the context?)
2. **User Feedback** (Did the user click thumbs up/down?)
3. **Toxicity / PII Leakage**

In [ ]:
eval_template = """
Read the Generated Answer and the Retrieved Context.
Are ALL claims in the Generated Answer directly supported by the Retrieved Context?
Output '1' for Yes, '0' for No.

Context: {context}
Answer: {answer}
"""
eval_prompt = ChatPromptTemplate.from_template(eval_template)
eval_llm = FakeListLLM(responses=["1", "0"]) # 1st trace faithful, 2nd trace unfaithful (hypothetically)
eval_chain = eval_prompt | eval_llm | StrOutputParser()

print("--- Running Asynchronous Evaluator ---")
for i, trace in enumerate(tracer.traces):
    score = eval_chain.invoke({"context": trace["context"], "answer": trace["answer"]})
    print(f"Trace {i+1} Faithfulness Score: {score}")

## Reflection

1. **Online vs Offline:** Offline evaluation uses `Context Recall` and `Answer Relevance` because you have a known Ground Truth. Online evaluation uses `Faithfulness` and `User Feedback` because you don't know the exact right answer.
2. **Feedback Loops:** If a production trace gets a 'thumbs down' from a user, or a '0' from the Faithfulness evaluator, you should route that trace to a human reviewer. The human fixes the answer, and that trace is added to your Offline Golden Dataset to prevent the regression from happening again.